In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt

# ── Config ──
MODEL_SUBDIR = "Qwen/Qwen2.5-14B-Instruct"

USER = os.environ.get("USER", os.environ.get("USERNAME", ""))
BASE = f"/work/pi_dagarwal_umass_edu/project_4/file_storage/{USER}"

PIPELINE_ROOT = f"{BASE}/experiment_outputs"
EVAL_ROOT     = f"{BASE}/evaluation_outputs"
ENTITY_ROOT   = f"{BASE}/entity_extraction_output"
VIZ_ROOT      = "visualization_outputs"

VIZ_DIR = f"{VIZ_ROOT}/{MODEL_SUBDIR}/agentic_rag"
os.makedirs(VIZ_DIR, exist_ok=True)

COLOR = "#9467bd"  # single series color for agentic_rag
LABEL = "Agentic RAG"

In [ ]:
# ── Load data ──
with open(f"{PIPELINE_ROOT}/{MODEL_SUBDIR}/local_agentic_rag.json", "r", encoding="utf-8") as f:
    pipeline_data = json.load(f)

with open(f"{EVAL_ROOT}/{MODEL_SUBDIR}/local_agentic_rag_eval.json", "r", encoding="utf-8") as f:
    eval_data = json.load(f)

with open(f"{ENTITY_ROOT}/{MODEL_SUBDIR}/local_agentic_rag_entity_results.json", "r", encoding="utf-8") as f:
    entity_data = json.load(f)

print(f"Questions in pipeline output : {len(pipeline_data['questions'])}")
print(f"Questions in eval output     : {len(eval_data['questions'])}")
print(f"Questions in entity output   : {len(entity_data['questions'])}")

In [ ]:
# ── Helpers ──

def is_collapsed(iteration_runs):
    """True if every run in the iteration has an identical canonical entity set."""
    sets = [frozenset(r["canonical_entities"]) for r in iteration_runs]
    return len(set(sets)) == 1

def mean_metric_per_round(experiment, metric_key):
    """Mean of metric_key across questions per iteration, from eval-style files."""
    max_iters = max(len(q["iterations"]) for q in experiment["questions"])
    round_means = []
    for iter_num in range(max_iters):
        vals = [
            q["iterations"][iter_num]["metrics"].get(metric_key, 0.0)
            for q in experiment["questions"]
            if iter_num < len(q["iterations"])
        ]
        round_means.append(np.mean(vals) if vals else 0.0)
    return round_means

In [ ]:
# ── Collapse (bar chart) ──

def compute_collapse_metrics(experiment):
    start_collapsed, end_collapsed, pct_rounds = [], [], []
    for q in experiment["questions"]:
        iters = q["iterations"]
        if not iters:
            continue
        flags = [is_collapsed(it["runs"]) for it in iters]
        start_collapsed.append(1 if flags[0] else 0)
        end_collapsed.append(1 if flags[-1] else 0)
        pct_rounds.append(sum(flags) / len(flags) * 100)
    n = len(start_collapsed)
    return {
        "start_mean":  sum(start_collapsed) / n * 100,
        "end_mean":    sum(end_collapsed)   / n * 100,
        "end_std":     np.std([c * 100 for c in end_collapsed]),
        "rounds_mean": np.mean(pct_rounds),
        "rounds_std":  np.std(pct_rounds),
    }

m = compute_collapse_metrics(entity_data)
width = 0.25
x = np.array([0])

fig, ax = plt.subplots(figsize=(6, 6))
b1 = ax.bar(x - width, [m["start_mean"]],  width, label="% Collapsed at Start", color="#1f77b4")
b2 = ax.bar(x,         [m["end_mean"]],    width, yerr=[m["end_std"]],    capsize=4, label="% Collapsed at End",   color="#ff7f0e")
b3 = ax.bar(x + width, [m["rounds_mean"]], width, yerr=[m["rounds_std"]], capsize=4, label="% Rounds Collapsed",   color="#2ca02c")

for bar, color in [(b1, "#1f77b4"), (b2, "#ff7f0e"), (b3, "#2ca02c")]:
    for rect in bar:
        ax.text(rect.get_x() + rect.get_width() / 2, rect.get_height() + 1,
                f"{rect.get_height():.1f}", ha="center", va="bottom", fontsize=9, fontweight="bold", color=color)

ax.set_ylabel("%", fontsize=12)
ax.set_title("Collapse — Agentic RAG", fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(["agentic_rag\nsimulation"], fontsize=10)
ax.legend(loc="upper right")
ax.set_ylim(0, 115)

plt.tight_layout()
plt.savefig(f"{VIZ_DIR}/collapse.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Unique Words Per Round ──

uw = mean_metric_per_round(eval_data, "unique_words")

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(range(1, len(uw) + 1), uw, linewidth=2.5, label=LABEL, color=COLOR)
ax.set_xlabel("Round", fontsize=12)
ax.set_ylabel("Unique Words", fontsize=12)
ax.set_title("Unique Words Per Round — Agentic RAG", fontsize=14)
ax.legend()

plt.tight_layout()
plt.savefig(f"{VIZ_DIR}/unique_words_per_round.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Unique Entities Per Round ──

agg = entity_data["aggregate_statistics"]["mean_unique_entities_per_round"]

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(range(1, len(agg) + 1), agg, linewidth=2.5, label=LABEL, color=COLOR)
ax.set_xlabel("Round", fontsize=12)
ax.set_ylabel("Unique Entities", fontsize=12)
ax.set_title("Unique Entities Per Round — Agentic RAG", fontsize=14)
ax.legend()

plt.tight_layout()
plt.savefig(f"{VIZ_DIR}/unique_entities_per_round.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Entity Similarity Per Round ──

agg = entity_data["aggregate_statistics"]["mean_entity_similarity_per_round"]

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(range(1, len(agg) + 1), agg, linewidth=2.5, label=LABEL, color=COLOR)
ax.set_xlabel("Round", fontsize=12)
ax.set_ylabel("Entity Similarity", fontsize=12)
ax.set_title("Entity Similarity Per Round — Agentic RAG", fontsize=14)
ax.legend()

plt.tight_layout()
plt.savefig(f"{VIZ_DIR}/entity_similarity_per_round.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Same Answer Percentage Per Round ──

sap = mean_metric_per_round(eval_data, "same_answer_percentage")

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(range(1, len(sap) + 1), sap, linewidth=2.5, label=LABEL, color=COLOR)
ax.set_xlabel("Round", fontsize=12)
ax.set_ylabel("Same Answer Percentage (%)", fontsize=12)
ax.set_title("Same Answer Percentage Per Round — Agentic RAG", fontsize=14)
ax.set_ylim(max(0, min(sap) - 5), 101)
ax.legend()

plt.tight_layout()
plt.savefig(f"{VIZ_DIR}/same_answer_percentage_per_round.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Avg Pairwise TES Per Round ──

tes = mean_metric_per_round(eval_data, "avg_pairwise_tes")

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(range(1, len(tes) + 1), tes, linewidth=2.5, label=LABEL, color=COLOR)
ax.set_xlabel("Round", fontsize=12)
ax.set_ylabel("Avg Pairwise TES", fontsize=12)
ax.set_title("Avg Pairwise TES Per Round — Agentic RAG", fontsize=14)
ax.set_ylim(max(0, min(tes) - 0.05), min(1.0, max(tes) + 0.05))
ax.legend()

plt.tight_layout()
plt.savefig(f"{VIZ_DIR}/avg_pairwise_tes_per_round.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Avg Pairwise ROUGE-1 Per Round ──

r1 = mean_metric_per_round(eval_data, "avg_pairwise_rouge1")

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(range(1, len(r1) + 1), r1, linewidth=2.5, label=LABEL, color=COLOR)
ax.set_xlabel("Round", fontsize=12)
ax.set_ylabel("Avg Pairwise ROUGE-1", fontsize=12)
ax.set_title("Avg Pairwise ROUGE-1 Per Round — Agentic RAG", fontsize=14)
ax.set_ylim(max(0, min(r1) - 0.05), min(1.0, max(r1) + 0.05))
ax.legend()

plt.tight_layout()
plt.savefig(f"{VIZ_DIR}/avg_pairwise_rouge1_per_round.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Avg Pairwise ROUGE-2 Per Round ──

r2 = mean_metric_per_round(eval_data, "avg_pairwise_rouge2")

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(range(1, len(r2) + 1), r2, linewidth=2.5, label=LABEL, color=COLOR)
ax.set_xlabel("Round", fontsize=12)
ax.set_ylabel("Avg Pairwise ROUGE-2", fontsize=12)
ax.set_title("Avg Pairwise ROUGE-2 Per Round — Agentic RAG", fontsize=14)
ax.set_ylim(max(0, min(r2) - 0.05), min(1.0, max(r2) + 0.05))
ax.legend()

plt.tight_layout()
plt.savefig(f"{VIZ_DIR}/avg_pairwise_rouge2_per_round.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Avg Pairwise ROUGE-L Per Round ──

rl = mean_metric_per_round(eval_data, "avg_pairwise_rougeL")

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(range(1, len(rl) + 1), rl, linewidth=2.5, label=LABEL, color=COLOR)
ax.set_xlabel("Round", fontsize=12)
ax.set_ylabel("Avg Pairwise ROUGE-L", fontsize=12)
ax.set_title("Avg Pairwise ROUGE-L Per Round — Agentic RAG", fontsize=14)
ax.set_ylim(max(0, min(rl) - 0.05), min(1.0, max(rl) + 0.05))
ax.legend()

plt.tight_layout()
plt.savefig(f"{VIZ_DIR}/avg_pairwise_rougeL_per_round.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Avg Tool Calls Per Round (line chart) ──
# tool_calls_used is stored per run inside the raw pipeline output.
# For each iteration round, average tool_calls_used across all runs of all questions.

def mean_tool_calls_per_round(pipeline_experiment):
    max_iters = max(len(q["iterations"]) for q in pipeline_experiment["questions"])
    round_means = []
    for iter_num in range(max_iters):
        counts = [
            run["tool_calls_used"]
            for q in pipeline_experiment["questions"]
            if iter_num < len(q["iterations"])
            for run in q["iterations"][iter_num]["runs"]
            if "tool_calls_used" in run
        ]
        round_means.append(np.mean(counts) if counts else 0.0)
    return round_means

tc = mean_tool_calls_per_round(pipeline_data)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(range(1, len(tc) + 1), tc, linewidth=2.5, label=LABEL, color=COLOR)
ax.set_xlabel("Round", fontsize=12)
ax.set_ylabel("Avg Tool Calls", fontsize=12)
ax.set_title("Avg Tool Calls Per Round — Agentic RAG", fontsize=14)
ax.set_ylim(0, max(tc) + 0.5)
ax.legend()

plt.tight_layout()
plt.savefig(f"{VIZ_DIR}/avg_tool_calls_per_round.png", dpi=150, bbox_inches="tight")
plt.show()